Using the EETC to verify optimal exposure times. This notebook should use the corgiloop environment specified by the corgihowfsc install.

In [1]:
from eetc.cgi_eetc import CGIEETC
import eetc
import numpy as np
import yaml
from astropy.table import Table
import os
from pathlib import Path

/home/jarenashcraft/anaconda3/envs/corgiloop/lib/python3.12/site-packages/astropy/config/paths.py:55: AstropyUserWarning: XDG_CONFIG_HOME is set to '/home/jarenashcraft/.var/app/com.vscodium.codium/config', but the default location, /home/jarenashcraft/.astropy/config, already exists, and takes precedence. This environment variable will be ignored.
  return set_temp_config._get_dir_path(rootname)


Set up the root path to the `eetc` repository to find the observation yaml files

In [2]:
eetc_root = eetc.lib_dir
pointer_path_sci = os.path.join(eetc_root, 'pointer_howfsc.yaml')

with open(pointer_path_sci, 'r') as file:
    pointer_data = yaml.safe_load(file)

Check EXCAMM properties

In [3]:
# let's print out the excam config properties being assumed in the optimizer
excam_config_file = os.path.join(eetc_root, pointer_data['excam_config'])
with open(excam_config_file, 'r') as file:
    excam_data = yaml.safe_load(file)

print('EXCAM parameters assumed for exposure optimization:\n')

keys_to_print = ['alpha0','alpha1','tmin','tmax','overhead','pc_ecount_max']

notes_to_print = ['fraction of pixel full well to allowed',
                  'fraction of serial full well to allow',
                  'minimum exposure time [s]','maximum exposure time [s]',
                  'per frame overhead [s]',
                  'Max allowed e-/pixel/frame for photon counting']

for ii, key in enumerate(keys_to_print):
    print(key,': ', excam_data[key], '(' + notes_to_print[ii] + ')')

# also get an idea of the maximum photon counting flux rate allowed by the EETC optimizer
print('\n------------------------')
print('Max flux rate allowed for photon counting mode in focal plane [e-/pix/s]: %.4f'%(excam_data['pc_ecount_max']/excam_data['tmin']))
print('------------------------')


EXCAM parameters assumed for exposure optimization:

alpha0 :  0.75 (fraction of pixel full well to allowed)
alpha1 :  0.75 (fraction of serial full well to allow)
tmin :  0.1 (minimum exposure time [s])
tmax :  120.0 (maximum exposure time [s])
overhead :  12 (per frame overhead [s])
pc_ecount_max :  0.25 (Max allowed e-/pixel/frame for photon counting)

------------------------
Max flux rate allowed for photon counting mode in focal plane [e-/pix/s]: 2.5000
------------------------


Look at sequence definitions

In [4]:
# file containing all of the sequences available

sequence_file = Path.home() / "corgi_ObsPlanning/eetc_sequences/sequences.yaml"
with open(sequence_file, 'r') as file:
    data = yaml.safe_load(file)

rows = []
for config_name, params in data.items():
    # Ensure params is a dictionary (for robustness)
    if isinstance(params, dict):
        row = {'config_name': config_name}
        row.update(params)
        rows.append(row)

# 3. Create the Astropy Table
config_table = Table(rows=rows)
config_table.add_index('config_name')
print(config_table.keys())

['config_name', 'mode', 'dms', 'spam_lsam', 'fpam', 'fsam', 'cfam', 'dpam', 'peak_flux_ratio_pix', 'num_pixels', 'fraction']


Setting up the polarized sequence

In [5]:
# Get the etc helper function
from helper_functions import rough_wall_clock

# BGPS atlas directory
atlas_dir = os.path.join(eetc_root,'flux_grid_generation','bpgs_atlas_csv')
spt_type_files = [sp.split('.txt')[0] for sp in os.listdir(atlas_dir)]
print(spt_type_files)
seq_name = ""

['A0IV', 'A0V', 'A1V', 'A2V', 'A3III', 'A3IV', 'A3V', 'A4IV', 'A4V', 'A5III', 'A5IV', 'A5V', 'A7V', 'A9IV', 'A9V', 'B0IB', 'B1IV', 'B1V', 'B2III', 'B2V', 'B3III', 'B3IV', 'B3V', 'B4V', 'B5IB', 'B6V', 'B7III', 'B7IV', 'B7V', 'B8IA', 'B9IV', 'B9V', 'CVS', 'CVS2', 'CVS3', 'CVS4', 'F0IV', 'F0V', 'F2IV', 'F2V', 'F4V', 'F5IV', 'F5V', 'F6V', 'F7IV', 'F7V', 'F8V', 'F9V', 'G0V', 'G2IV', 'G2V', 'G3IV', 'G3V', 'G4IV', 'G4V', 'G5IV', 'G5V', 'G6IV', 'G6V', 'G7IV', 'G7V', 'G8III', 'G8IV', 'G8V', 'K0III', 'K0IV', 'K0V', 'K1III', 'K1IV', 'K1V', 'K2III', 'K2IIIP', 'K2IV', 'K2V', 'K3III', 'K3V', 'K4III', 'K4V', 'K5III', 'K7V', 'K8V', 'M0III', 'M0V', 'M1III', 'M2III', 'M2V', 'M3III', 'M4V', 'M5III', 'M5V', 'M6III', 'M6V', 'M7III', 'M8', 'M8III', 'M8V', 'NEPTUNE', 'O5', 'O6', 'O8F', 'URANUS']


In [6]:
# set up your star parameters
star_mag = 1.69 
star_spt = 'A3IV'
star_mag_filter = 'v'

# set up flux ratio for your object of interest parameters. 
# Flux ratio is in the observation band, not V band.
# single star or when companion/disk is fainter than speckles: use the raw contrast value
# point source: use its flux ratio *if* it is brighter than the speckle field
# disk: use its flux ratio per resolution element *if* it is brighter than speckle field
flux_ratio = 1 # this value is the multiplicative factor applied to the 'raw' stellar flux rate
snr_desired = 5

In [7]:
# initialize eetc object
cgi_eetc = CGIEETC(mag=star_mag, 
                   phot=star_mag_filter, 
                   spt=star_spt,
                   pointer_path=pointer_path_sci)

seq_name = "OPEN_NFOV_WOLL_1B_OPTDMS"

In [8]:
# unocculted, on-axis flux rate
total_flux_rate_unocc, peak_flux_rate_unocc = cgi_eetc.calc_flux_rate(sequence_name=seq_name)
print('Total unocculted target flux rate in focal plane [e-/s]: %.1f'%total_flux_rate_unocc)
print('Peak unocculted pixel flux rate in focal plane [e-/s]: %.1f'%peak_flux_rate_unocc)
print('-------------')


Total unocculted target flux rate in focal plane [e-/s]: 249781528.0
Peak unocculted pixel flux rate in focal plane [e-/s]: 3659055.1
-------------


In [ ]:
# Let's use an ND-filtered version of the sequence selected above since this star is bright
contrast = 3e-8
bright_scaling = 1e-7
snr_desired = 5
num_frames, exp_time_frame, gain, snr_out, optflag = \
    cgi_eetc.calc_exp_time(sequence_name=seq_name, 
                                 snr=snr_desired,
                                 scale=contrast,
                                 scale_bright=bright_scaling)

names = ['num_frames', 'exp_time_frame', 'gain', 'snr_out']

print('\n----------------')
print('DERIVED EXPOSURE SETTINGS')
for name in names:
    print(name+': %.3f'%eval(name))
print('----------------\n')

# now we can compute the total wall clock time using the helper function defined above
wall_t = rough_wall_clock(n_frames=num_frames, exp_t=exp_time_frame)
print('rough wall clock time [s]: %.3f'%wall_t)

Optimization scheme used:  optflag = 0
target SNR per pixel:  5
SNR per pixel output from optimization:  5.5768055647118



----------------
DERIVED EXPOSURE SETTINGS
num_frames: 5.000
exp_time_frame: 120.000
gain: 921.864
snr_out: 5.577
----------------

rough wall clock time [s]: 604.320
